# FAISS Project – SIFT Starter Experiment

This notebook loads the SIFT dataset, builds exact ground truth, and runs a first IVFPQ experiment with Recall@10 and QPS.

In [ ]:
import sys
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

In [ ]:
from src.data_loading import load_sift
from src.benchmarks import build_ground_truth, benchmark_index
from src.metrics import recall_at_k

## Load SIFT

In [ ]:
xb, xq, xlearn, gt = load_sift()

print("xb:", xb.shape, xb.dtype)
print("xq:", xq.shape, xq.dtype)
print("xlearn:", xlearn.shape, xlearn.dtype)
print("gt:", gt.shape, gt.dtype)

## Exact Ground Truth

In [ ]:
k = 10
d = xb.shape[1]
metric = faiss.METRIC_L2

D_gt, I_gt = build_ground_truth(xb, xq, k, metric)
print("D_gt shape:", D_gt.shape)
print("I_gt shape:", I_gt.shape)

## First IVFPQ Experiment

In [ ]:
factory = "IVF4096,PQ32x8"
index = faiss.index_factory(d, factory, metric)

train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb
if not index.is_trained:
    index.train(train_data)

index.add(xb)

In [ ]:
ps = faiss.ParameterSpace()
ps.set_index_parameter(index, "nprobe", 16)

D, I, total_time, qps = benchmark_index(index, xq, k)
recall = recall_at_k(I, I_gt, k)

print("Factory:", factory)
print("Recall@10:", recall)
print("QPS:", qps)
print("Total search time:", total_time)

## Notes

This is the first working benchmark. Next steps are parameter sweeps for IVFPQ, HNSWSQ, and LSH.

## IVFPQ Sweep on SIFT

In [ ]:
from src.indexes import set_ivfpq_nprobe, set_hnswsq_efsearch

ivfpq_factories = ["IVF4096,PQ8x8", "IVF4096,PQ16x8", "IVF4096,PQ32x8"]
nprobe_values = [1, 4, 8, 16, 32, 64]

In [ ]:
ivfpq_rows = []
train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb

for factory in ivfpq_factories:
    index = faiss.index_factory(d, factory, metric)
    if not index.is_trained:
        index.train(train_data)
    index.add(xb)

    for nprobe in nprobe_values:
        set_ivfpq_nprobe(index, int(nprobe))
        D, I, total_time, qps = benchmark_index(index, xq, k)
        recall = recall_at_k(I, I_gt, k)

        ivfpq_rows.append({
            "dataset": "sift",
            "method": "IVFPQ",
            "factory": factory,
            "search_param_name": "nprobe",
            "search_param_value": nprobe,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
ivfpq_results_df = pd.DataFrame(ivfpq_rows)
ivfpq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
ivfpq_path = tables_dir / "sift_ivfpq_results.csv"
ivfpq_results_df.to_csv(ivfpq_path, index=False)
print(f"Saved: {ivfpq_path}")

## HNSWSQ Sweep on SIFT

In [ ]:
hnsw_factories = ["HNSW32_SQ4", "HNSW32_SQ6", "HNSW32_SQ8"]
efsearch_values = [16, 32, 64, 128, 256]

In [ ]:
hnswsq_rows = []
train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb

for factory in hnsw_factories:
    index = faiss.index_factory(d, factory, metric)
    if not index.is_trained:
        index.train(train_data)
    index.add(xb)

    for efsearch in efsearch_values:
        set_hnswsq_efsearch(index, int(efsearch))
        D, I, total_time, qps = benchmark_index(index, xq, k)
        recall = recall_at_k(I, I_gt, k)

        hnswsq_rows.append({
            "dataset": "sift",
            "method": "HNSWSQ",
            "factory": factory,
            "search_param_name": "efSearch",
            "search_param_value": efsearch,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
hnswsq_results_df = pd.DataFrame(hnswsq_rows)
hnswsq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
hnswsq_path = tables_dir / "sift_hnswsq_results.csv"
hnswsq_results_df.to_csv(hnswsq_path, index=False)
print(f"Saved: {hnswsq_path}")

## LSH Sweep on SIFT

In [ ]:
lsh_bits = [128, 256, 512]

In [ ]:
lsh_rows = []

for nbits in lsh_bits:
    index = faiss.IndexLSH(d, int(nbits))
    index.add(xb)

    D, I, total_time, qps = benchmark_index(index, xq, k)
    recall = recall_at_k(I, I_gt, k)

    lsh_rows.append({
        "dataset": "sift",
        "method": "LSH",
        "factory": "IndexLSH",
        "search_param_name": "nbits",
        "search_param_value": nbits,
        "recall_at_10": recall,
        "qps": qps,
        "total_time_sec": total_time,
    })

In [ ]:
lsh_results_df = pd.DataFrame(lsh_rows)
lsh_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
lsh_path = tables_dir / "sift_lsh_results.csv"
lsh_results_df.to_csv(lsh_path, index=False)
print(f"Saved: {lsh_path}")

## Combined SIFT Results

In [ ]:
sift_results_df = pd.concat([ivfpq_results_df, hnswsq_results_df, lsh_results_df], ignore_index=True)
sift_results_df.shape

In [ ]:
sift_results_sorted_df = sift_results_df.sort_values(["recall_at_10", "qps"], ascending=[False, False]).reset_index(drop=True)
sift_results_sorted_df

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
all_path = tables_dir / "sift_all_results.csv"
sift_results_df.to_csv(all_path, index=False)
print(f"Saved: {all_path}")

## SIFT Recall–QPS Plots

In [ ]:
plt.figure(figsize=(8, 5))

for method in ["IVFPQ", "HNSWSQ", "LSH"]:
    subset = sift_results_df[sift_results_df["method"] == method]
    plt.scatter(subset["qps"], subset["recall_at_10"], label=method, alpha=0.8)

plt.xlabel("QPS")
plt.ylabel("Recall@10")
plt.title("SIFT: Recall@10 vs QPS")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
plot_path = plots_dir / "sift_recall_vs_qps.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {plot_path}")

## Best SIFT Configurations

In [ ]:
top_recall_df = sift_results_df.sort_values(["recall_at_10", "qps"], ascending=[False, False]).head(5).copy()
top_recall_df["summary"] = "top_recall"

top_qps_df = sift_results_df.sort_values(["qps", "recall_at_10"], ascending=[False, False]).head(5).copy()
top_qps_df["summary"] = "top_qps"

best_sift_summary_df = pd.concat([top_recall_df, top_qps_df], ignore_index=True)[
    ["summary", "method", "factory", "search_param_name", "search_param_value", "recall_at_10", "qps", "total_time_sec"]
]
best_sift_summary_df

## SIFT Experiment Notes

- Next step: RC analysis.
- Next step: hard/easy query groups.
- Next step: noisy query robustness evaluation.
- Next step: repeat the same scaffold for GIST and then GloVe.